In [99]:
import pandas as pd
import numpy as np


In [100]:
df  = pd.read_csv("Resume.csv")
df

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR
...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [101]:
df = df[['Resume_str', 'Category']]

In [102]:
# Remove missing values
df.dropna(inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

print("After cleaning:", df.shape)

After cleaning: (2482, 2)


C:\Users\indiranivas_s\AppData\Local\Temp\ipykernel_26916\423062783.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)
C:\Users\indiranivas_s\AppData\Local\Temp\ipykernel_26916\423062783.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [103]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Run once (if not downloaded)
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Clean function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\indiranivas_s\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\indiranivas_s\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [104]:

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

C:\Users\indiranivas_s\AppData\Local\Temp\ipykernel_26916\657633658.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cleaned_resume'] = df['Resume_str'].apply(clean_text)


In [105]:
df.head()

,Resume_str,Category,cleaned_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist u hr operation summary versatile...
2,HR DIRECTOR Summary Over 2...,HR,hr director summary year experience recruiting...
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlight hr skill hr departm...


In [106]:
import datetime

In [107]:
pip install python-dateutil

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
import re
from datetime import datetime
from dateutil import parser

def extract_experience(text):
    text = text.lower()
    current_date = datetime.now()

    # -------- 1. Direct years --------
    match = re.search(r'(\d+)\+?\s+year', text)
    if match:
        return float(match.group(1))

    total_days = 0

    # -------- 2. Find all date ranges --------
    date_ranges = re.findall(
        r'([a-zA-Z0-9/\s]+?)\s*(?:to|-)\s*([a-zA-Z0-9/\s]+)',
        text
    )

    for start, end in date_ranges:
        try:
            # Clean text
            start = start.strip()
            end = end.strip()

            # Handle current/present
            if "current" in end or "present" in end:
                end_date = current_date
            else:
                end_date = parser.parse(end, fuzzy=True)

            start_date = parser.parse(start, fuzzy=True)

            # Calculate difference
            diff = (end_date - start_date).days

            # Ignore unrealistic values
            if 0 < diff < 365 * 50:
                total_days += diff

        except:
            continue

    return round(total_days / 365, 1)

In [109]:
df['experience'] = df['Resume_str'].apply(extract_experience)

C:\Users\indiranivas_s\AppData\Local\Temp\ipykernel_26916\3333958612.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['experience'] = df['Resume_str'].apply(extract_experience)


In [110]:
df

,Resume_str,Category,cleaned_resume,experience
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...,15.0
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist u hr operation summary versatile...,16.2
2,HR DIRECTOR Summary Over 2...,HR,hr director summary year experience recruiting...,20.0
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...,20.0
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlight hr skill hr departm...,19.2
...,...,...,...,...
2478,ADVANCED LEVEL WHEELED VEHICLE MECHAN...,AVIATION,advanced level wheeled vehicle mechanic career...,11.7
2479,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,AVIATION,rank sgte non commissioned officer charge brig...,23.8
2480,"GOVERNMENT RELATIONS, COMMUNICATIONS ...",AVIATION,government relation communication organization...,15.4
2481,GEEK SQUAD AGENT Professional...,AVIATION,geek squad agent professional profile support ...,10.2


In [111]:
sum(df['experience']==0.0)

62

In [112]:
from transformers import pipeline

# Named Entity Recognition (NER)
ner = pipeline("ner", model="dslim/bert-base-NER")

def extract_skills_bert(text):
    entities = ner(text)
    
    skills = []
    
    for ent in entities:
        if ent['entity'].startswith('B-') or ent['entity'].startswith('I-'):
            skills.append(ent['word'])
    
    return list(set(skills))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2717.29it/s]
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [113]:
text = "Experienced in Python, TensorFlow and data analysis"

print(extract_skills_bert(df['Resume_str'][0]))

['Custom', 'and', '##ity', 'OR', '##A', 'Reservation', '##S', 'Missouri', '##lide', '##s', 'Micro', '##G', '##del', 'Worldwide', 'Fi', 'Opera', '##H', '##P', 'Ho', 'O', 'System', '##Q', 'I', 'State', 'Hospital', '##io', 'On', 'PM', 'Hilton', 'Management', '##ER', '##er', 'Service', '##x']


In [114]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [115]:
def calculate_match_score(resume, job_description):
    
    # Combine texts
    documents = [resume, job_description]
    
    # TF-IDF Vectorization
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    # Cosine similarity
    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
    
    # Convert to percentage
    score = similarity[0][0] * 100
    
    return round(score, 2)

In [116]:

resume = df['cleaned_resume'][2]

job_description = """
We are looking for a skilled Python Developer to join our team and build 
high-quality software solutions. The ideal candidate should have strong 
programming skills, experience in backend development, and the ability 
to work with databases and APIs.

Responsibilities:
- Develop, test, and maintain Python-based applications
- Write clean, efficient, and reusable code
- Work with databases and perform data processing tasks
- Design and develop REST APIs and backend services
- Debug and resolve software issues
- Collaborate with team members to deliver high-quality solutions
- Ensure application performance and scalability

Required Skills:
- Strong proficiency in Python programming
- Good understanding of data structures and algorithms
- Experience with SQL and database management
- Familiarity with libraries such as pandas and numpy
- Experience in API development (Flask / FastAPI)
- Knowledge of version control systems like Git

Preferred Skills:
- Experience with web frameworks like Django or Flask
- Familiarity with cloud platforms (AWS, Azure, GCP)
- Understanding of software development lifecycle
- Experience with Docker or deployment tools

Experience:
- 2+ years of experience in Python development
- Experience working on real-world projects

Education:
- Bachelor’s degree in Computer Science, IT, or related field
"""

print(calculate_match_score(resume, job_description))

9.02


In [117]:
df["score"] = df["cleaned_resume"].apply(
    lambda resume: calculate_match_score(resume, job_description)
)

C:\Users\indiranivas_s\AppData\Local\Temp\ipykernel_26916\727300435.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["score"] = df["cleaned_resume"].apply(


In [120]:
df_pass=df[df["score"]>=20]
df_pass

,Resume_str,Category,cleaned_resume,experience,score
1762,ENGINEERING AND QUALITY TECHNICIAN ...,ENGINEERING,engineering quality technician career overview...,3.0,20.34
